In [1]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

# Agora você pode importar os módulos da pasta scripts usando a notação de ponto
from scripts.crm_leads import load_leads
from scripts.crm_inscricoes import load_inscricoes





In [2]:
inscricoes = load_inscricoes()

Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 200 registros


In [3]:
inscricoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Numero Inscricao          3200 non-null   str  
 1   Nome Completo             3200 non-null   str  
 2   Situação de Avaliação     3200 non-null   str  
 3   Situação Inscrição        3200 non-null   str  
 4   Curso ID                  3200 non-null   str  
 5   Unidade Inscrito          515 non-null    str  
 6   Forma de Ingresso         3200 non-null   str  
 7   createdon                 3200 non-null   str  
 8   cad_inscricaocandidatoid  3200 non-null   str  
 9   Data Aprovação            1061 non-null   str  
 10  Situação Pagamento        1256 non-null   str  
 11  Data de Pagamento         658 non-null    str  
dtypes: str(12)
memory usage: 300.1 KB


In [33]:
leads = load_leads()


Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 173 registros


In [34]:
"""
CRM Educacional -> Power BI
Script refatorado (template)

Preencha a lista `atributos` e o dicionário `rename_columns`
com os valores do seu projeto.
"""

import time
import requests
import xmltodict
import pandas as pd

BASE_URL = "https://trivento.api.crmeducacional.com"
URL = f"{BASE_URL}/api/LocalizacaoAvancada"

USERNAME = "IntegraçãoCRM"
PASSWORD = "IntegraçãoCRM"
AUTH_TYPE = "APICRMEducacional"

atributos = [
    'cad_numeroinscricao',
    "cad_clientepotencialid",
    "crmeduc_data_aprovacao",
    "cad_datapagamento",
    "cad_situacaoavaliacao",
    "cad_situacao",
    "cad_ofertaid",
    "trivento_unidadedeensinotrivento",
    "cad_concursoid",
    "coc_situacaopagamento",
]

rename_columns = {
    'cad_inscricaocandidatoid': 'id_inscricao',
    "cad_clientepotencialid": "ID Lead",
    'cad_numeroinscricao': 'Numero Inscricao',
    "crmeduc_data_aprovacao": "Data Aprovação",
    "cad_datapagamento": "Data de Pagamento",
    "cad_situacaoavaliacao": "Situação de Avaliação",
    "cad_situacao": "Situação Inscrição",
    "cad_ofertaid": "Curso ID",
    "trivento_unidadedeensinotrivento": "Unidade Inscrito",
    "cad_concursoid": "Forma de Ingresso",
    "coc_situacaopagamento": "Situação Pagamento",
}


def get_token(session):
    payload = {
        "grant_type": "password",
        "username": USERNAME,
        "password": PASSWORD,
        "authentication_type": AUTH_TYPE,
    }

    response = session.post(
        f"{BASE_URL}/api/token",
        data=payload,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def create_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/xml",
    }


def create_fetch_xml(attributes):
    tags = "\n".join(
        f'<attribute name="{c}"/>' for c in attributes
    )

    return f"""
<fetch version="1.0" output-format="xml-platform" mapping="logical">
  <entity name="cad_inscricaocandidato">
    {tags}
    <order attribute="createdon" descending="true"/>
  </entity>
</fetch>
"""


def post_retry(session, *args, retries=3, **kwargs):
    for i in range(retries):
        try:
            r = session.post(*args, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if i == retries - 1:
                raise
            time.sleep(5)


def get_page(session, page, headers, body):
    headers = headers.copy()
    headers["Pagina"] = str(page)

    response = post_retry(
        session,
        URL,
        headers=headers,
        data=body.encode("utf-8"),
        timeout=60,
    )

    doc = xmltodict.parse(response.text)

    resultset = doc["resultset"]
    records = resultset["result"]

    if isinstance(records, dict):
        records = [records]

    return records, resultset["@morerecords"] == "1"


def get_all_records(session, headers, body):
    page = 1
    rows = []

    while True:
        records, more = get_page(session, page, headers, body)
        rows.extend(records)
        print(f"Página {page}: {len(records)} registros")

        if not more:
            break

        page += 1

    return rows


def flatten_value(value):
    if not isinstance(value, dict):
        return value

    if "@name" in value:
        return value["@name"]

    if "#text" in value:
        return value["#text"]

    return value


def flatten_record(record):
    return {
        k: v['#text'] if k == 'cad_clientepotencialid' else  flatten_value(v)
        for k, v in record.items()
    }


def load_inscricoes():

    with requests.Session() as session:

        token = get_token(session)
        headers = create_headers(token)
        body = create_fetch_xml(atributos)

        records = get_all_records(
            session,
            headers,
            body,
        )
    
    df = pd.DataFrame(
        flatten_record(r)
        for r in records
    )

    for col in ["'createdon'", "modifiedon"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                format="%d/%m/%Y %H:%M",
                errors="coerce",
            )

    df.rename(
        columns={
            k: v
            for k, v in rename_columns.items()
            if k in df.columns
        },
        inplace=True,
    )

    map = {
        "Faculdade Serra Dourada de " : "",
        "Faculdade Alis" : "Itabirito",
        " dos Carajás" : ""}
    
    df['Unidade Inscrito'] = df['Unidade Inscrito'].map(map)
    return df

inscricoes = load_inscricoes()





Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 152 registros


In [35]:
inscricoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3152 entries, 0 to 3151
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Numero Inscricao       3152 non-null   object
 1   ID Lead                3152 non-null   object
 2   Situação de Avaliação  3152 non-null   object
 3   Situação Inscrição     3152 non-null   object
 4   Curso ID               3152 non-null   object
 5   Unidade Inscrito       504 non-null    object
 6   Forma de Ingresso      3152 non-null   object
 7   createdon              3152 non-null   object
 8   id_inscricao           3152 non-null   object
 9   Data Aprovação         1053 non-null   object
 10  Situação Pagamento     1251 non-null   object
 11  Data de Pagamento      657 non-null    object
dtypes: object(12)
memory usage: 295.6+ KB


In [36]:
print(f'Total de leads: {leads.shape[0]}')
print(f'Total de inscricoes: {inscricoes.shape[0]}')

Total de leads: 6173
Total de inscricoes: 3152


In [37]:
leads = leads[['ID Lead', 'Nome Completo']]

In [38]:
inscricoes = inscricoes[['ID Lead', 'id_inscricao']]

In [39]:
df = leads.merge(inscricoes, how= 'left', on = 'ID Lead')

In [40]:
df.head()

,ID Lead,Nome Completo,id_inscricao
0,{13F8C032-9788-F111-8076-6045BD38FB26},Lívia Cavalheiro,{8A7C0352-9788-F111-8076-6045BD38FB26}
1,{2C157D6C-7D88-F111-8077-6045BD39BC63},Heitor Leite,NaN
2,{66836F53-7C88-F111-8077-6045BD39BC63},Lídia oliveira,NaN
3,{BC030D8D-7B88-F111-8077-6045BD39BC63},Laniara silva silva,NaN
4,{C00217C4-7A88-F111-8077-6045BD39BE00},Ivani Esteves de Melo,NaN


In [41]:
colunas = ['ID Lead','id_inscricao']

for coluna in colunas:
    df[coluna] = df[coluna].str.strip().str.lower()
    df[coluna] = df[coluna].str.replace('{', '').str.replace('}', '')

In [42]:
df['Nome Completo'] = df['Nome Completo'].str.strip().str.lower()

In [43]:
testes = df[df['Nome Completo'].str.contains('teste')]

In [44]:
testes.shape

(19, 3)

In [45]:
testes.sample(10)

,ID Lead,Nome Completo,id_inscricao
3267,62067e57-9671-f111-ab0f-000d3ac0f764,gleison abcds teste,6e9f765d-9671-f111-ab0f-000d3ac0f764
3272,d4ed5aea-8b71-f111-ab0f-7ced8db11fb7,teste teste teste,2573fbfc-8b71-f111-ab0f-7ced8db11fb7
2340,5fe43e2b-8e76-f111-ab0e-7ced8db11fb7,teste teste,6ba75d31-8e76-f111-ab0e-7ced8db11fb7
3315,0e476e18-7671-f111-ab0f-7ced8db11fb7,teste bridha,b93e691e-7671-f111-ab0f-7ced8db11fb7
3500,52435328-f36f-f111-ab0e-6045bd39bc63,daniel teste de pagamento,02771041-f36f-f111-ab0e-6045bd39bc63
3383,3028a12a-dc70-f111-ab0f-00224836b0b2,teste pag car,2ea68531-dc70-f111-ab0f-00224836b0b2
3403,e4eb67bc-d270-f111-ab0e-6045bd39bc63,teste pag cartao,17f289e0-d270-f111-ab0f-6045bd39be00
3435,e9ded011-a470-f111-ab0f-7ced8db11fb7,carol teste ficha,eddfd011-a470-f111-ab0f-7ced8db11fb7
4054,5579a1e8-3969-f111-ab0c-6045bd39bc63,benitez alteração em massa teste,NaN
3438,efdddacd-a170-f111-ab0e-6045bd39bc63,teste corporativo 2,NaN


In [46]:
testes.to_csv('dataset.csv', index= False)

In [47]:
testes.loc[testes['ID Lead'] == '69e6a318-007a-f111-ab0e-000d3ac0f764', 'id_inscricao'].values

array([], dtype=object)